# Data Cleaning Pipeline

Applies all findings from the EDA to produce `train_clean.csv` and `test_clean.csv`.

**Steps**
1. Load raw data
2. Drop useless columns (`User_ID`, `Payment_Schedule`, `Policy_Start_Day`)
3. Handle missing values
4. Feature engineering (17 new features)
5. Categorical encoding (ordinal / binary / one-hot / target)
6. Final validation & save

> Run this notebook once; then use `train_clean.csv` / `test_clean.csv` in the modelling notebook.

## 1. Imports & Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import warnings, os
warnings.filterwarnings('ignore')

DATA_DIR  = '../Data'
OUT_DIR   = '../Data'
SEED      = 42
N_SPLITS  = 5       # folds for OOF target encoding
SMOOTHING = 300     # smoothing weight for target encoding

print('pandas', pd.__version__)
print('numpy ', np.__version__)

pandas 3.0.1
numpy  2.4.2


## 2. Load Raw Data

In [2]:
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print('Train:', train_raw.shape)
print('Test: ', test_raw.shape)
train_raw.head(3)

Train: (60868, 29)
Test:  (15218, 28)


,User_ID,Policy_Cancelled_Post_Purchase,Policy_Start_Year,Policy_Start_Week,Policy_Start_Day,Grace_Period_Extensions,Previous_Policy_Duration_Months,Adult_Dependents,Child_Dependents,Infant_Dependents,...,Custom_Riders_Requested,Broker_Agency_Type,Deductible_Tier,Acquisition_Channel,Payment_Schedule,Employment_Status,Estimated_Annual_Income,Days_Since_Quote,Policy_Start_Month,Purchased_Coverage_Bundle
0,USR_000000,1,2016,50,8,0,3,2,0.0,0,...,0,Urban_Boutique,Tier_4_Zero_Ded,Aggregator_Site,Monthly_EFT,Employed_FullTime,26267.93,28,April,2
1,USR_000001,0,2016,10,4,0,2,2,0.0,0,...,0,National_Corporate,Tier_1_High_Ded,Direct_Website,Monthly_EFT,Employed_FullTime,24355.18,4,July,4
2,USR_000002,0,2016,9,22,1,0,1,0.0,0,...,0,National_Corporate,Tier_1_High_Ded,Aggregator_Site,Monthly_EFT,Employed_FullTime,13033.18,0,June,2


## 3. Drop Useless Columns

| Column | Reason |
| --- | --- |
| `User_ID` | Identifier only — no predictive value |
| `Payment_Schedule` | 98.7% `Monthly_EFT` — zero variance |
| `Policy_Start_Day` | Correlation with target = 0.013 — noise |

In [3]:
DROP_COLS = ['User_ID', 'Payment_Schedule', 'Policy_Start_Day']

train = train_raw.drop(columns=DROP_COLS)
# test has no target column; drop same columns (minus target if absent)
test_drop = [c for c in DROP_COLS if c in test_raw.columns]
test  = test_raw.drop(columns=test_drop)

print('Train after drop:', train.shape)
print('Test  after drop:', test.shape)

Train after drop: (60868, 26)
Test  after drop: (15218, 25)


## 4. Handle Missing Values

| Column | Strategy | Reason |
| --- | --- | --- |
| `Child_Dependents` | Fill 0 | Only 4 nulls; children absent = 0 |
| `Deductible_Tier` | Fill mode (`Tier_1_High_Ded`) | 314 nulls; dominant class |
| `Region_Code` | Fill `UNKNOWN` | 303 nulls; preserve as distinct category |
| `Acquisition_Channel` | Fill `Aggregator_Site` | 666 nulls; dominant class |
| `Broker_ID` | Leave NaN | Target encoder handles unseen; flag created separately |
| `Employer_ID` | Leave NaN | Binary flag captures signal; numeric value unused |

In [4]:
def fill_missing(df):
    df = df.copy()
    df['Child_Dependents']   = df['Child_Dependents'].fillna(0)
    df['Deductible_Tier']    = df['Deductible_Tier'].fillna('Tier_1_High_Ded')
    df['Region_Code']        = df['Region_Code'].fillna('UNKNOWN')
    df['Acquisition_Channel']= df['Acquisition_Channel'].fillna('Aggregator_Site')
    return df

train = fill_missing(train)
test  = fill_missing(test)

# Confirm no unexpected nulls remain
null_counts = train.isnull().sum()
print('Remaining nulls in train:')
print(null_counts[null_counts > 0].to_string())

Remaining nulls in train:
Broker_ID       8331
Employer_ID    57396


## 5. Feature Engineering

### 5.1 Dependents

In [5]:
def add_dependent_features(df):
    df = df.copy()
    df['Total_Dependents'] = (
        df['Adult_Dependents']
        + df['Child_Dependents'].fillna(0)
        + df['Infant_Dependents']
    )
    df['Has_Children'] = (
        (df['Child_Dependents'].fillna(0) + df['Infant_Dependents']) > 0
    ).astype(int)
    return df

train = add_dependent_features(train)
test  = add_dependent_features(test)
print('Total_Dependents mean:', train['Total_Dependents'].mean().round(3))
print('Has_Children rate:   ', train['Has_Children'].mean().round(3))

Total_Dependents mean: 2.051
Has_Children rate:    0.11


### 5.2 Binary Flags

In [6]:
def add_binary_flags(df):
    df = df.copy()
    # Employer presence — 94.3% missing; flag captures group-sponsored policies
    df['Employer_ID_Present']  = df['Employer_ID'].notna().astype(int)
    # Broker missing — data completeness signal
    df['Broker_ID_Missing']    = df['Broker_ID'].isna().astype(int)
    # Underwriting delay — median=0; binary flag more useful than raw days
    df['Underwriting_Delayed'] = (df['Underwriting_Processing_Days'] > 0).astype(int)
    # Fast buyer (<7 days quote-to-purchase) skews toward Basic_Health
    df['Fast_Buyer']           = (df['Days_Since_Quote'] < 7).astype(int)
    return df

train = add_binary_flags(train)
test  = add_binary_flags(test)
print('Employer_ID_Present rate:', train['Employer_ID_Present'].mean().round(4))
print('Broker_ID_Missing rate:  ', train['Broker_ID_Missing'].mean().round(4))
print('Underwriting_Delayed rate:', train['Underwriting_Delayed'].mean().round(4))
print('Fast_Buyer rate:         ', train['Fast_Buyer'].mean().round(4))

Employer_ID_Present rate: 0.057
Broker_ID_Missing rate:   0.1369
Underwriting_Delayed rate: 0.0094
Fast_Buyer rate:          0.1723


### 5.3 Income Transforms

In [7]:
def add_income_features(df):
    df = df.copy()
    # Log-transform: income max ($1.8M) is 48x the mean — extreme right skew
    df['log_Income'] = np.log1p(df['Estimated_Annual_Income'])
    # Per-person affordability proxy
    df['Income_Per_Dependent'] = (
        df['Estimated_Annual_Income'] / (df['Total_Dependents'] + 1)
    )
    df['log_Income_Per_Dependent'] = np.log1p(df['Income_Per_Dependent'])
    return df

train = add_income_features(train)
test  = add_income_features(test)
print('log_Income stats:')
print(train['log_Income'].describe().round(3))

log_Income stats:
count    60868.000
mean        10.237
std          1.555
min          0.000
25%         10.134
50%         10.460
75%         10.780
max         14.424
Name: log_Income, dtype: float64


### 5.4 Risk Composite Score

In [8]:
def add_risk_features(df):
    df = df.copy()
    # Risk_Score: higher = more financial stress / risk
    # (+) claims filed, (+) grace extensions, (-) years clean
    df['Risk_Score'] = (
        df['Previous_Claims_Filed']
        + df['Grace_Period_Extensions']
        - df['Years_Without_Claims']
    )
    return df

train = add_risk_features(train)
test  = add_risk_features(test)
print('Risk_Score distribution:')
print(train['Risk_Score'].value_counts().head(8))

Risk_Score distribution:
Risk_Score
 0    22733
 2    18454
 1    15290
 4     1248
 3     1043
-1      667
-2      315
-3      195
Name: count, dtype: int64


### 5.5 Interaction & Complexity Features

In [9]:
def add_interaction_features(df):
    df = df.copy()
    # Indecision: slow buyer who also kept amending = complex purchase
    df['Indecision_Score']  = df['Days_Since_Quote'] * df['Policy_Amendments_Count']
    # Overall policy complexity
    df['Policy_Complexity'] = df['Custom_Riders_Requested'] + df['Policy_Amendments_Count']
    return df

train = add_interaction_features(train)
test  = add_interaction_features(test)
print('Indecision_Score mean:', train['Indecision_Score'].mean().round(2))
print('Policy_Complexity mean:', train['Policy_Complexity'].mean().round(2))

Indecision_Score mean: 26.23
Policy_Complexity mean: 0.98


### 5.6 Temporal — Month Name → Integer

In [10]:
MONTH_ORDER = {
    'January':1, 'February':2, 'March':3,    'April':4,
    'May':5,     'June':6,     'July':7,      'August':8,
    'September':9,'October':10,'November':11, 'December':12
}

def add_temporal_features(df):
    df = df.copy()
    df['Month_Num'] = df['Policy_Start_Month'].map(MONTH_ORDER)
    # Cyclical encoding for month (period = 12)
    df['Month_Sin'] = np.sin(2 * np.pi * df['Month_Num'] / 12)
    df['Month_Cos'] = np.cos(2 * np.pi * df['Month_Num'] / 12)
    # Cyclical encoding for week (period = 52)
    df['Week_Sin']  = np.sin(2 * np.pi * df['Policy_Start_Week'] / 52)
    df['Week_Cos']  = np.cos(2 * np.pi * df['Policy_Start_Week'] / 52)
    return df

train = add_temporal_features(train)
test  = add_temporal_features(test)
print('Month_Num value counts:')
print(train['Month_Num'].value_counts().sort_index())

Month_Num value counts:
Month_Num
1     4665
2     4776
3     3441
4     3588
5     3182
6     4218
7     5216
8     5562
9     5808
10    5446
11    7036
12    7930
Name: count, dtype: int64


## 6. Categorical Encoding

### 6.1 Ordinal Encoding — `Deductible_Tier`

Higher tier = lower deductible = higher premium. Natural ordinal order.

In [11]:
DEDUCTIBLE_ORDER = {
    'Tier_1_High_Ded': 1,   # highest deductible / lowest premium
    'Tier_2_Mid_Ded':  2,
    'Tier_3_Low_Ded':  3,
    'Tier_4_Zero_Ded': 4,   # zero deductible / highest premium
}

train['Deductible_Tier_Ord'] = train['Deductible_Tier'].map(DEDUCTIBLE_ORDER)
test['Deductible_Tier_Ord']  = test['Deductible_Tier'].map(DEDUCTIBLE_ORDER)

# Fill any unmapped / still-null values with mode (Tier_1=1)
train['Deductible_Tier_Ord'] = train['Deductible_Tier_Ord'].fillna(1).astype(int)
test['Deductible_Tier_Ord']  = test['Deductible_Tier_Ord'].fillna(1).astype(int)

print('Deductible_Tier ordinal distribution:')
print(train['Deductible_Tier_Ord'].value_counts().sort_index())

Deductible_Tier ordinal distribution:
Deductible_Tier_Ord
1    48002
2     6455
3      258
4     6153
Name: count, dtype: int64


### 6.2 Binary Encoding — `Broker_Agency_Type`

Only 2 values → single binary column.

In [12]:
# Urban_Boutique=0, National_Corporate=1
AGENCY_MAP = {'Urban_Boutique': 0, 'National_Corporate': 1}

train['Agency_National'] = train['Broker_Agency_Type'].map(AGENCY_MAP)
test['Agency_National']  = test['Broker_Agency_Type'].map(AGENCY_MAP)

print('Agency_National value counts:')
print(train['Agency_National'].value_counts())

Agency_National value counts:
Agency_National
0    36829
1    24039
Name: count, dtype: int64


### 6.3 One-Hot Encoding — `Employment_Status` & `Acquisition_Channel`

In [13]:
# Employment_Status: 4 values — drop first to avoid multicollinearity
emp_dummies_train = pd.get_dummies(
    train['Employment_Status'], prefix='Emp', drop_first=True
).astype(int)
emp_dummies_test  = pd.get_dummies(
    test['Employment_Status'],  prefix='Emp', drop_first=True
).astype(int)

# Align columns (in case test is missing a category)
emp_dummies_test = emp_dummies_test.reindex(
    columns=emp_dummies_train.columns, fill_value=0
)

train = pd.concat([train, emp_dummies_train], axis=1)
test  = pd.concat([test,  emp_dummies_test],  axis=1)
print('Employment dummies:', list(emp_dummies_train.columns))

# Acquisition_Channel: 5 values — drop first
acq_dummies_train = pd.get_dummies(
    train['Acquisition_Channel'], prefix='Acq', drop_first=True
).astype(int)
acq_dummies_test  = pd.get_dummies(
    test['Acquisition_Channel'],  prefix='Acq', drop_first=True
).astype(int)
acq_dummies_test = acq_dummies_test.reindex(
    columns=acq_dummies_train.columns, fill_value=0
)

train = pd.concat([train, acq_dummies_train], axis=1)
test  = pd.concat([test,  acq_dummies_test],  axis=1)
print('Acquisition dummies:', list(acq_dummies_train.columns))

Employment dummies: ['Emp_Employed_FullTime', 'Emp_Self_Employed', 'Emp_Unemployed']
Acquisition dummies: ['Acq_Aggregator_Site', 'Acq_Corporate_Partner', 'Acq_Direct_Website', 'Acq_Local_Broker']


### 6.4 Target Encoding — `Broker_ID`, `Region_Code`, `Employer_ID`

**Why target encoding?**  These columns have high cardinality (315 / 166 / 309 unique values) so one-hot would add hundreds of sparse columns. Target encoding compresses each category into one float — the smoothed mean of the target within that group.

**Smoothing formula:** `smoothed = (n × mean + k × global_mean) / (n + k)` where `k=300`. Rare categories shrink toward the global mean.

**Leakage prevention:** Train values are computed with **Out-Of-Fold** encoding (5-fold StratifiedKFold). Test values use the full-train smoothed mean.

In [14]:
TARGET = 'Purchased_Coverage_Bundle'

def smoothed_mean(series, target_series, global_mean, smoothing):
    """Compute smoothed target mean for a categorical series."""
    stats = pd.DataFrame({'cat': series, 'target': target_series})
    agg   = stats.groupby('cat')['target'].agg(['mean', 'count'])
    agg['smoothed'] = (
        (agg['count'] * agg['mean'] + smoothing * global_mean)
        / (agg['count'] + smoothing)
    )
    return agg['smoothed']


def target_encode_oof(train_df, col, target_col, n_splits=5, smoothing=300):
    """OOF target encoding for train set — no leakage."""
    global_mean = train_df[target_col].mean()
    oof_enc     = pd.Series(np.nan, index=train_df.index)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for fold_tr_idx, fold_val_idx in skf.split(train_df, train_df[target_col]):
        fold_tr  = train_df.iloc[fold_tr_idx]
        fold_val = train_df.iloc[fold_val_idx]
        mapping  = smoothed_mean(
            fold_tr[col], fold_tr[target_col], global_mean, smoothing
        )
        oof_enc.iloc[fold_val_idx] = fold_val[col].map(mapping).fillna(global_mean)

    return oof_enc


def target_encode_test(train_df, test_df, col, target_col, smoothing=300):
    """Full-train smoothed target encoding for test set."""
    global_mean = train_df[target_col].mean()
    mapping     = smoothed_mean(
        train_df[col], train_df[target_col], global_mean, smoothing
    )
    return test_df[col].map(mapping).fillna(global_mean)


print('Target encoding functions defined.')

Target encoding functions defined.


In [15]:
# ── Broker_ID ──────────────────────────────────────────────────────────────
# Fill NaN with a sentinel before encoding so OOF can map them too
train['Broker_ID_filled'] = train['Broker_ID'].fillna(-1)
test['Broker_ID_filled']  = test['Broker_ID'].fillna(-1)

train['Broker_ID_TargetEnc'] = target_encode_oof(
    train, 'Broker_ID_filled', TARGET, N_SPLITS, SMOOTHING
)
test['Broker_ID_TargetEnc']  = target_encode_test(
    train, test, 'Broker_ID_filled', TARGET, SMOOTHING
)

# ── Region_Code ────────────────────────────────────────────────────────────
train['Region_TargetEnc'] = target_encode_oof(
    train, 'Region_Code', TARGET, N_SPLITS, SMOOTHING
)
test['Region_TargetEnc']  = target_encode_test(
    train, test, 'Region_Code', TARGET, SMOOTHING
)

# ── Employer_ID ────────────────────────────────────────────────────────────
# Encode only non-null rows; nulls → global mean (captured by binary flag too)
train['Employer_ID_filled'] = train['Employer_ID'].fillna(-1)
test['Employer_ID_filled']  = test['Employer_ID'].fillna(-1)

train['Employer_ID_TargetEnc'] = target_encode_oof(
    train, 'Employer_ID_filled', TARGET, N_SPLITS, SMOOTHING
)
test['Employer_ID_TargetEnc']  = target_encode_test(
    train, test, 'Employer_ID_filled', TARGET, SMOOTHING
)

print('Target encoding complete.')
print('Broker_ID_TargetEnc sample:')
print(train[['Broker_ID','Broker_ID_TargetEnc']].drop_duplicates().dropna().head(8).to_string())

Target encoding complete.
Broker_ID_TargetEnc sample:
   Broker_ID  Broker_ID_TargetEnc
0        9.0             2.827699
1      250.0             3.660579
2      240.0             2.728583
3        1.0             2.237877
4      240.0             2.731333
6       40.0             2.977524
7        9.0             2.836791
8      170.0             2.727103


## 7. Drop Raw / Intermediate Columns

Remove originals that have been replaced by engineered features.

In [16]:
RAW_TO_DROP = [
    # Already one-hot encoded
    'Employment_Status',
    'Acquisition_Channel',
    # Already ordinally / binary encoded
    'Deductible_Tier',
    'Broker_Agency_Type',
    # Replaced by Month_Num + cyclical features
    'Policy_Start_Month',
    # Replaced by binary flag + target encoding
    'Broker_ID',
    'Employer_ID',
    # Sentinel fill columns (internal only)
    'Broker_ID_filled',
    'Employer_ID_filled',
    # Raw income replaced by log transform
    'Estimated_Annual_Income',
    # Income_Per_Dependent replaced by log version
    'Income_Per_Dependent',
    # Region_Code replaced by target encoding
    'Region_Code',
]

# Only drop columns that actually exist
to_drop_train = [c for c in RAW_TO_DROP if c in train.columns]
to_drop_test  = [c for c in RAW_TO_DROP if c in test.columns]

train = train.drop(columns=to_drop_train)
test  = test.drop(columns=to_drop_test)

print('Train shape after drop:', train.shape)
print('Test  shape after drop:', test.shape)

Train shape after drop: (60868, 45)
Test  shape after drop: (15218, 44)


## 8. Validation

In [17]:
# Check for any remaining nulls
null_train = train.isnull().sum()
null_test  = test.isnull().sum()

print('=== TRAIN — remaining nulls ===')
if null_train.any():
    print(null_train[null_train > 0].to_string())
else:
    print('None')

print()
print('=== TEST — remaining nulls ===')
if null_test.any():
    print(null_test[null_test > 0].to_string())
else:
    print('None')

=== TRAIN — remaining nulls ===
None

=== TEST — remaining nulls ===
None


In [18]:
# Column inventory
print(f'Train: {train.shape[0]:,} rows, {train.shape[1]} columns')
print(f'Test:  {test.shape[0]:,} rows, {test.shape[1]} columns')
print()

# Check train and test have the same feature columns
train_feats = set(train.columns) - {TARGET}
test_feats  = set(test.columns)
only_train  = train_feats - test_feats
only_test   = test_feats  - train_feats

if only_train:
    print('Only in train (expected: target):', sorted(only_train))
if only_test:
    print('Only in test  (unexpected):', sorted(only_test))

print()
print('All feature columns:')
feat_cols = sorted(train_feats)
for i, c in enumerate(feat_cols, 1):
    print(f'  {i:2}. {c}')

Train: 60,868 rows, 45 columns
Test:  15,218 rows, 44 columns


All feature columns:
   1. Acq_Aggregator_Site
   2. Acq_Corporate_Partner
   3. Acq_Direct_Website
   4. Acq_Local_Broker
   5. Adult_Dependents
   6. Agency_National
   7. Broker_ID_Missing
   8. Broker_ID_TargetEnc
   9. Child_Dependents
  10. Custom_Riders_Requested
  11. Days_Since_Quote
  12. Deductible_Tier_Ord
  13. Emp_Employed_FullTime
  14. Emp_Self_Employed
  15. Emp_Unemployed
  16. Employer_ID_Present
  17. Employer_ID_TargetEnc
  18. Existing_Policyholder
  19. Fast_Buyer
  20. Grace_Period_Extensions
  21. Has_Children
  22. Indecision_Score
  23. Infant_Dependents
  24. Month_Cos
  25. Month_Num
  26. Month_Sin
  27. Policy_Amendments_Count
  28. Policy_Cancelled_Post_Purchase
  29. Policy_Complexity
  30. Policy_Start_Week
  31. Policy_Start_Year
  32. Previous_Claims_Filed
  33. Previous_Policy_Duration_Months
  34. Region_TargetEnc
  35. Risk_Score
  36. Total_Dependents
  37. Underwriting_Delayed
  38.

In [19]:
# Dtype summary — all should be numeric
non_numeric = train.select_dtypes(include='object').columns.tolist()
if non_numeric:
    print('WARNING — object columns still present:', non_numeric)
else:
    print('All columns are numeric.')

print()
print('Dtype counts:')
print(train.dtypes.value_counts())

All columns are numeric.

Dtype counts:
int64      34
float64    11
Name: count, dtype: int64


In [20]:
# Quick sanity check on target distribution
print('Target distribution (train):')
print(train[TARGET].value_counts().sort_index().to_string())

Target distribution (train):
Purchased_Coverage_Bundle
0      823
1     1625
2    36136
3     4831
4    13958
5      479
6      719
7     2286
8        6
9        5


## 9. Save Cleaned Datasets

In [21]:
train_out = os.path.join(OUT_DIR, 'train_clean.csv')
test_out  = os.path.join(OUT_DIR, 'test_clean.csv')

train.to_csv(train_out, index=False)
test.to_csv(test_out,  index=False)

print(f'Saved train_clean.csv  → {train.shape}')
print(f'Saved test_clean.csv   → {test.shape}')
print()
print('Preview — train_clean head(3):')
train.head(3)

Saved train_clean.csv  → (60868, 45)
Saved test_clean.csv   → (15218, 44)

Preview — train_clean head(3):


,Policy_Cancelled_Post_Purchase,Policy_Start_Year,Policy_Start_Week,Grace_Period_Extensions,Previous_Policy_Duration_Months,Adult_Dependents,Child_Dependents,Infant_Dependents,Existing_Policyholder,Previous_Claims_Filed,...,Emp_Employed_FullTime,Emp_Self_Employed,Emp_Unemployed,Acq_Aggregator_Site,Acq_Corporate_Partner,Acq_Direct_Website,Acq_Local_Broker,Broker_ID_TargetEnc,Region_TargetEnc,Employer_ID_TargetEnc
0,1,2016,50,0,3,2,0.0,0,0,0,...,1,0,0,1,0,0,0,2.827699,2.763301,2.772102
1,0,2016,10,0,2,2,0.0,0,0,0,...,1,0,0,0,0,1,0,3.660579,2.624635,2.771876
2,0,2016,9,1,0,1,0.0,0,0,0,...,1,0,0,1,0,0,0,2.728583,2.624635,2.771876
